# Gemini Inference

Clean notebook for Gemini-based ratings or critiques runs using the shared schedules created in the OpenAI pipeline.

## Imports And Notebook Root

In [ ]:
import os
import sys
import json
from pathlib import Path
import pandas as pd

ROOT_CODE_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils").exists())
NOTEBOOK_HOME = ROOT_CODE_DIR / "experiment_notebooks" / "Gemini_Pipeline"
os.chdir(NOTEBOOK_HOME)

if str(ROOT_CODE_DIR) not in sys.path:
    sys.path.append(str(ROOT_CODE_DIR))

from utils.llm_evaluation_utils import *
from utils.prompt_builder import build_rating_prompt, build_critique_prompt
from utils.data_setup import get_dataset_file_path, prepare_project_data
from utils.models_setup import setup_gemini, query_gemini_model
from utils.gemini_batch_manager import GeminiBatchManager

ROOT_CODE_DIR, NOTEBOOK_HOME


## Config And Constants

In [ ]:
TASK_SUBSET = "1000_tasks"      # "all_tasks" | "1000_tasks" | "50_tasks"
PROMPTING_TYPE = "zero_shot"   # "zero_shot" | "few_shot"
STAGE = "critiques"            # "ratings" | "critiques"
NUM_TRIALS = 1
MAX_TOKENS = 30000
TEMPERATURE = 1
MODEL_VERSION = "gemini-2.5-pro"

TASK_SUBSET_MAP = {
    "all_tasks": "all",
    "1000_tasks": "1000",
    "50_tasks": "50",
}
DATA_TASK_SUBSET = TASK_SUBSET_MAP[TASK_SUBSET]

if STAGE == "ratings":
    SYSTEM_MESSAGE = RATING_SYSTEM_MESSAGE
    build_prompt_fn = build_rating_prompt
    update_results_df_fn = update_rating_in_df
elif STAGE == "critiques":
    SYSTEM_MESSAGE = CRITIQUES_SYSTEM_MESSAGE
    build_prompt_fn = build_critique_prompt
    update_results_df_fn = update_critiques_in_df
else:
    raise ValueError(f"Invalid stage: {STAGE}")

client, cfg = setup_gemini(
    model=MODEL_VERSION,
    system_message=SYSTEM_MESSAGE,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

cfg


## Data Loading

In [ ]:
uicrit_file, base64_screens_file, few_shot_samples_file = get_dataset_file_path(STAGE)

uicrit_df = pd.read_parquet(uicrit_file)
base64_screens_df = pd.read_parquet(base64_screens_file)

if few_shot_samples_file:
    few_shot_samples_df = pd.read_parquet(few_shot_samples_file)
else:
    few_shot_samples_df = None

print("UICrit rows:", len(uicrit_df))
print("Screens rows:", len(base64_screens_df))


## Shared Schedules

In [ ]:
schedule_dir = ROOT_CODE_DIR / "experiment_notebooks" / "OpenAI_Pipeline" / "schedules"

rating_schedule_file_1000_tasks = schedule_dir / "rating_trial_schedule-1000_tasks.parquet"
rating_schedule_file_all_tasks = schedule_dir / "rating_trial_schedule-all_tasks.parquet"
critiques_schedule_file_1000_tasks = schedule_dir / "critiques_trial_schedule-1000_tasks.parquet"
critiques_schedule_file_all_tasks = schedule_dir / "critiques_trial_schedule-all_tasks.parquet"
critiques_schedule_file_all_tasks_excluding_1000 = schedule_dir / "critiques_trial_schedule-all_tasks_excluding_1000.parquet"

print("Using shared schedule dir:", schedule_dir)


## Select Schedule For The Current Run

In [ ]:
if TASK_SUBSET == "1000_tasks":
    if STAGE == "ratings":
        schedule_file = rating_schedule_file_1000_tasks
    else:
        schedule_file = critiques_schedule_file_1000_tasks
elif TASK_SUBSET == "all_tasks":
    if STAGE == "ratings":
        schedule_file = rating_schedule_file_all_tasks
    else:
        schedule_file = critiques_schedule_file_all_tasks
elif TASK_SUBSET == "50_tasks":
    if STAGE == "ratings":
        schedule_file = rating_schedule_file_all_tasks
    else:
        schedule_file = critiques_schedule_file_all_tasks
else:
    raise ValueError(f"Invalid TASK_SUBSET: {TASK_SUBSET}")

print("Using schedule file:", schedule_file)


## Load Responses DataFrame

In [ ]:
responses_df, results_paths = prepare_project_data(
    model_short=cfg["model_short"],
    num_trials=NUM_TRIALS,
    task_subset=DATA_TASK_SUBSET,
    shots=PROMPTING_TYPE,
    stage=STAGE,
)

text_responses_jsonl_file = results_paths["results_jsonl"]
model_results_file = results_paths["results_parquet"]

print("Responses shape:", responses_df.shape)
responses_df.head(2)


## Batch Setup

In [ ]:
batch_main_dir = NOTEBOOK_HOME / f"{STAGE}_Batching" / f"Batch_Files-{TASK_SUBSET}-{PROMPTING_TYPE}-gemini"
batch_main_dir.mkdir(parents=True, exist_ok=True)

batch_process = GeminiBatchManager(
    client=client,
    batch_main_dir=batch_main_dir,
    selected_tasks=TASK_SUBSET,
)

batch_main_dir


## Prepare Prompts JSONL

In [ ]:
prompts_file = batch_process.generate_prompts_jsonl(
    schedule_file=schedule_file,
    responses_df=responses_df,
    screens_df=base64_screens_df,
    model_version=cfg["model_version"],
    system_message=SYSTEM_MESSAGE,
    guidelines=GUIDELINES,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    prompting_type=PROMPTING_TYPE,
    samples_df=few_shot_samples_df,
    image_format="jpeg",
    build_prompt_fn=build_prompt_fn,
)

prompts_file


In [ ]:
with open(prompts_file, "r", encoding="utf-8") as f:
    for _ in range(4):
        print(json.loads(f.readline()))


In [ ]:
batch_process.split_prompts_jsonl(max_lines_per_file=200)


## Batch Requests

Use the manual section when you want fine-grained control. Use the automatic section when you want the notebook to upload, create, poll, and download sequentially.

### Manual Batch Workflow

In [ ]:
batch_file_path = batch_process.prompts_dir / "prompts-batch_1.jsonl"
input_file_id = batch_process.upload_file(batch_file_path)
input_file_id


In [ ]:
_ = batch_process.list_uploaded_files()


In [ ]:
batch = batch_process.create_batch(
    input_file_id=input_file_id,
    model_version=cfg["model_version"],
)
batch


In [ ]:
_ = batch_process.list_batches()


In [ ]:
state, batch = batch_process.check_batch(verbose=True)
state


In [ ]:
responses_file_path = batch_process.retrieve_batch_output(batch_obj=batch, base_name="responses")
responses_file_path


### Automatic Batch Workflow

In [ ]:
start_from_batch = 1
resume_batch = None

batch_process.process_batches_from(
    start_from_batch=start_from_batch,
    sleep_minutes=2,
    resume_batch_id=resume_batch,
)


## Extract Results From Batch Responses

In [ ]:
summary = batch_process.extract_results_from_responses(
    responses_df,
    update_results_df_fn,
)

print(
    "Updated:", summary["updated"],
    "| Skipped:", len(summary["skipped"]),
    "| Errors:", len(summary["errors"]),
)

responses_df.head(2)


In [ ]:
responses_dir = Path(batch_process.responses_dir)
output_file = Path(batch_process.prompts_file).parent / f"responses-{TASK_SUBSET}.jsonl"

count = 0
with output_file.open("w", encoding="utf-8") as out_f:
    for p in sorted(responses_dir.iterdir()):
        if not p.is_file():
            continue
        if p.resolve() == output_file.resolve():
            continue
        text = p.read_text(encoding="utf-8", errors="ignore")
        if text:
            if not text.endswith("\n"):
                text += "\n"
            out_f.write(text)
            count += 1

print(f"Written {count} files to {output_file}")


## Alternative: One-By-One Requests

Use this section instead of batching when you want to test prompts or run a very small sample interactively.

In [ ]:
responses_df_sample = responses_df.head(1).copy()
responses_df_sample


In [ ]:
query_args = dict(
    client=client,
    model_version=cfg["model_version"],
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    system=SYSTEM_MESSAGE,
    image_format="jpeg",
    config=cfg["config"],
)

run_llm_inference(
    responses_df=responses_df_sample,
    base64_screens_df=base64_screens_df,
    few_shot_samples_df=few_shot_samples_df,
    evaluation_aspects=EVALUATION_MAIN_ASPECTS,
    build_prompt_fn=build_prompt_fn,
    update_results_df_fn=update_results_df_fn,
    query_fn=query_gemini_model,
    query_args=query_args,
    save_jsonl_fn=save_response_text,
    output_jsonl=text_responses_jsonl_file,
    guidelines=GUIDELINES,
    prompting_type=PROMPTING_TYPE,
    requests_per_minute=cfg["rpm"],
    stage=STAGE,
    print_output=True,
)


## Explore Results

In [ ]:
responses_df.tail(5)


In [ ]:
if STAGE == "critiques":
    columns_with_none = (responses_df.isna() | (responses_df == "")).sum()
else:
    columns_with_none = responses_df[EVALUATION_FIVE_ASPECTS].isna().sum()

columns_with_none


In [ ]:
if STAGE == "critiques":
    rows_with_none = responses_df[responses_df["critiques"].isna() | (responses_df["critiques"] == "")]
else:
    rows_with_none = responses_df[responses_df[EVALUATION_FIVE_ASPECTS].isna().any(axis=1)]

rows_with_none.head()


In [ ]:
if STAGE == "critiques":
    missing = (responses_df["critiques"].isna() | (responses_df["critiques"] == "")).sum()
else:
    missing = responses_df[EVALUATION_FIVE_ASPECTS].isna().any(axis=1).sum()

print("rows:", len(responses_df))
print("unique screen_task_id:", responses_df["screen_task_id"].nunique())
print("missing outputs:", missing)


## Save Results

In [ ]:
responses_df.to_parquet(model_results_file, index=False)
print("Saved:", model_results_file)
